# Lecture 4 — Class Exercise
## Scatter & Bubble Charts: Gapminder

> **Push to:** `week04/lecture04_exercise.ipynb`

**Rules:**
1. Colour used **sparingly** — one categorical variable, no rainbow
2. If showing all continents, either use accessible palette OR grey all + highlight one
3. `size_max` set when using bubble size
4. Log scale for GDP per capita
5. Insight title

---


In [ ]:
import pandas as pd
import plotly.express as px


# Dataset: Gapminder — GDP, Life Expectancy, Population by Country
# Source: Gapminder Foundation (gapminder.org)

df = px.data.gapminder()
print(f"Loaded: {len(df)} rows")
print(df.head())

In [ ]:
# explore

print(df.info())
print("Years:", sorted(df['year'].unique()))
print("Continents:", df['continent'].unique())
print(df.describe().round(1))


## Task 1 — Scatter: life expectancy change over time

**What to build:** A scatter showing **GDP per capita vs life expectancy** for **two years** (2002 and 2007) to show how both moved — use **colour for year** (just 2 colours), **one continent only**.

Choose any continent except Africa (that was the example). Highlight the change direction.

> 💡 Filter: `df.loc[df['continent'] == 'YOUR_CHOICE']` then filter years


In [ ]:
# Task 1 — Europe: GDP vs life expectancy in 2002 vs 2007

euro = df.loc[
    (df['continent'] == 'Europe') & (df['year'].isin([2002, 2007]))
].copy()

fig = px.scatter(
    data_frame=euro,
    x='gdpPercap',
    y='lifeExp',
    color='year',
    color_discrete_map={2002: '#A0A0A0', 2007: '#2E75B6'},
    hover_name='country',
    log_x=True,
    template='plotly_white',
    labels={
        'gdpPercap': 'GDP per Capita (log scale)',
        'lifeExp': 'Life Expectancy (years)',
        'year': 'Year',
    },
    title=(
        'Europe, 2002→2007: countries shifted up and right — '
        'both wealth and life expectancy improved'
    ),
    height=480,
)

fig.update_traces(marker=dict(size=10, opacity=0.85))
fig.update_layout(
    font=dict(family='Arial', size=13),
    legend_title_text='',
    xaxis=dict(showgrid=False),
    yaxis=dict(showgrid=False),
)
fig.add_annotation(
    x=0.02,
    y=0.98,
    xref='paper',
    yref='paper',
    text='Blue = 2007  ·  Grey = 2002',
    showarrow=False,
    xanchor='left',
    yanchor='top',
    font=dict(size=12, color='#555555'),
)
fig.show()


## Task 2 — Bubble chart: tell a story

**What to build:** A bubble chart (full 2007 dataset, all countries) where:
- x = GDP per capita (log scale)
- y = life expectancy
- size = population
- colour = ONE continent highlighted (your choice), all others grey
- At least one annotation explaining the highlighted group's story

> This is the grey-and-highlight technique applied to a bubble chart.


In [ ]:
# Task 2 — 2007 bubble chart: highlight Asia, grey everything else

df_2007 = df.loc[df['year'] == 2007].copy()
highlight = 'Asia'
colour_map = {
    c: ('#2E75B6' if c == highlight else '#EEEEEE')
    for c in df_2007['continent'].unique()
}

fig = px.scatter(
    data_frame=df_2007,
    x='gdpPercap',
    y='lifeExp',
    size='pop',
    size_max=55,
    color='continent',
    color_discrete_map=colour_map,
    hover_name='country',
    hover_data={'continent': False, 'pop': ':,.0f'},
    log_x=True,
    template='plotly_white',
    labels={
        'gdpPercap': 'GDP per Capita (log scale)',
        'lifeExp': 'Life Expectancy (years)',
        'continent': '',
        'pop': 'Population',
    },
    title=(
        'Asia in 2007 spans the full wealth spectrum — '
        'from low-income countries to Japan and the Gulf'
    ),
    height=520,
)

# draw grey continents first, Asia on top
fig.data = tuple(
    sorted(fig.data, key=lambda t: 0 if t.name != highlight else 1)
)

fig.update_traces(
    selector=dict(name=highlight),
    marker=dict(opacity=0.85, line=dict(width=0.5, color='white')),
)
fig.update_traces(
    selector=lambda t: t.name != highlight,
    marker=dict(opacity=0.55),
)

fig.add_annotation(
    x=1500,  # data coords (axis is log-scaled)
    y=52,
    text='Asia: wide spread<br>in wealth & health',
    showarrow=False,
    font=dict(color='#2E75B6', size=12, family='Arial'),
    bgcolor='white',
    bordercolor='#2E75B6',
    borderwidth=1,
    borderpad=4,
)

fig.update_layout(
    showlegend=False,
    font=dict(family='Arial', size=13),
    xaxis=dict(showgrid=False),
    yaxis=dict(showgrid=False),
)
fig.show()
